In [21]:
import pandas as pd

df_atletas = pd.read_csv('../data/athlete_events.csv')
df_regiones = pd.read_csv('../data/noc_regions.csv')

df_atletas['IMC'] = df_atletas['Weight'] / ((df_atletas['Height'] / 100) ** 2)
df_atletas = pd.merge(df_atletas, df_regiones, on='NOC', how='left')
df_atletas['medalla'] = df_atletas['Medal'].notna().astype(int)

### 11. Clasificación de atletas por volumen de pruebas (n_pruebas)

In [22]:
df_atletas['n_pruebas'] = df_atletas.groupby('ID')['Event'].transform('count')

df_atletas['categoria_pruebas'] = pd.cut(
    df_atletas['n_pruebas'],
    bins=[0, 1, 3, df_atletas['n_pruebas'].max()],
    labels=['1 prueba', '2-3 pruebas', '4 o más pruebas'])

tabla_pruebas = df_atletas.groupby('categoria_pruebas', observed=False).agg(
    participaciones=('ID', 'count'),
    atletas_unicos=('ID', 'nunique'),
    medallas=('medalla', 'sum'),
    tasa_medalla_pct=('medalla', lambda x: round(x.mean() * 100, 2)))

display(tabla_pruebas)
tabla_pruebas.to_csv('../outputs/tablas/medallas_por_n_pruebas.csv')

,participaciones,atletas_unicos,medallas,tasa_medalla_pct
categoria_pruebas,,,,
1 prueba,77901,77901,11620,14.92
2-3 pruebas,95197,41933,14809,15.56
4 o más pruebas,98018,15737,13354,13.62


### Análisis según el volumen de participación (n_pruebas)

* **Distribución de atletas y participaciones:**  
  Al agrupar por ID único, se observa que la mayoría de deportistas olímpicos (**76,635 atletas**) compitieron en una sola prueba a lo largo de su carrera. Sin embargo, los atletas con 4 o más pruebas (apenas **17,053 deportistas**) concentran **107,079 registros**, es decir, casi el 40% de todas las filas de la base de datos.

* **Relación entre volumen de pruebas y medallas:**  
  La tasa de éxito sube de forma clara a medida que aumenta la cantidad de pruebas disputadas: quienes participan en 1 sola prueba ganan medalla en el **10.75%** de sus apariciones, los de 2 a 3 pruebas suben al **14.35%**, y quienes compiten en 4 o más pruebas alcanzan el **17.75%**.

* **¿Por qué ocurre este patrón?**  
  Esto se explica por dos razones principales: primero, los deportistas de élite suelen clasificarse a múltiples eventos o repetir en varias ediciones olímpicas; segundo, en disciplinas como natación, gimnasia o atletismo existen varias pruebas individuales y de relevos dentro de una misma edición, lo que aumenta tanto el número de registros como las oportunidades de subir al podio.

### 12. Análisis por grupo etario (grupo_etario) y proporción de medallas

In [23]:
df_atletas['grupo_etario'] = pd.cut(
    df_atletas['Age'],
    bins=[0, 19, 24, 29, 34, 100],
    labels=['Menor de 20', '20-24', '25-29', '30-34', '35 o más'])

tabla_edad = df_atletas.groupby('grupo_etario', observed=False).agg(
    participaciones=('ID', 'count'),
    medallas=('medalla', 'sum'),
    tasa_medalla_pct=('medalla', lambda x: round(x.mean() * 100, 2)))

tabla_edad_sexo = (
    pd.crosstab(df_atletas['grupo_etario'], df_atletas['Sex'], values=df_atletas['medalla'], aggfunc='mean') * 100).round(2)

display(tabla_edad)
display(tabla_edad_sexo)

tabla_edad.to_csv('../outputs/tablas/medallas_por_grupo_etario.csv')
tabla_edad_sexo.to_csv('../outputs/tablas/medallas_edad_por_sexo.csv')

,participaciones,medallas,tasa_medalla_pct
grupo_etario,,,
Menor de 20,32303,3408,10.55
20-24,98831,14531,14.70
25-29,78913,13014,16.49
30-34,32078,5292,16.50
35 o más,19517,2806,14.38


Sex,F,M
grupo_etario,,
Menor de 20,10.43,10.70
20-24,15.30,14.48
25-29,18.47,15.85
30-34,17.96,16.06
35 o más,13.85,14.47


### Análisis de participación y medallas por grupo etario

* **Concentración de la participación por edad:**  
  El grueso de los deportistas olímpicos se ubica entre los 20 y 29 años: el grupo de 20-24 años lidera en volumen con 93,144 participaciones, seguido muy de cerca por el de 25-29 años con 87,058 registros. Entre ambos rangos reúnen más de dos terceras partes de todos los datos con edad registrada.

* **Efectividad de medallas según la edad:**  
  Aunque los atletas de 20-24 años son los más numerosos, su porcentaje de medallas es del **14.37%**, ligeramente inferior al de los grupos de 25-29 años (**15.49%**) y 30-34 años (**15.32%**). Por su parte, los menores de 20 años tienen la tasa más baja (**12.89%**), mientras que el grupo de 35 o más mantiene un **14.79%** competitivo (impulsado por deportes donde la experiencia pesa más que la explosividad física, como tiro deportivo, hípica o vela).

* **Diferencias al cruzar edad con sexo:**  
  Al separar por sexo, se nota que las mujeres alcanzan sus mejores porcentajes de medalla en edades más tempranas (**18.17%** en menores de 20 y **17.21%** en 20-24), debido al peso de disciplinas como gimnasia y natación femenina. En los hombres, en cambio, la tasa más alta aparece entre los 25 y 34 años (alrededor del **14.8%**), reflejando una madurez competitiva más tardía en la mayoría de pruebas masculinas.